# Proyecto BI – Purchasing | Sprint 1

**Proyecto:** Análisis de compras y proveedores de AdventureWorks  
**Sprint:** 1 – Extracción, preparación y análisis exploratorio de datos (EDA)  
**Integrantes:** [Miguel Ángel Ortega Hernandez, José Manuel Molina, Sebastian Agudelo]  
**Fecha:** [14/10/2026]


## 1. Objetivo del sprint

Obtener y preparar los datos relacionados con compras y proveedores de AdventureWorks para comprender su estructura y calidad, realizar el análisis exploratorio de datos (EDA) y dejar una base de trabajo para las siguientes etapas del proyecto, incluida la posible segmentación de proveedores mediante K-Means.

En este notebook se documentan la fuente de los datos, las tablas utilizadas y el proceso de preparación. Los hallazgos del EDA y las decisiones sobre las variables de segmentación se incorporarán cuando el equipo complete su análisis.

## 2. Fuente y alcance de los datos

Los datos se obtuvieron de la base de datos AdventureWorks mediante una conexión de Python a SQL Server. Las tablas se exportaron a archivos CSV para facilitar su revisión y análisis local.

### Tablas extraídas

| Tabla | Descripción general |
|---|---|
| `Purchasing.Vendor` | Información principal de los proveedores. |
| `Purchasing.PurchaseOrderHeader` | Información general de las órdenes de compra, proveedor, fechas y valores. |
| `Purchasing.PurchaseOrderDetail` | Líneas de detalle de las órdenes, productos, cantidades y precios. |
| `Purchasing.ProductVendor` | Relación entre proveedores y productos, con información como precio y tiempo promedio de entrega. |
| `Purchasing.ShipMethod` | Catálogo de métodos de envío. |
| `Production.Product` | Información descriptiva de los productos. |

### Archivos generados

- `data/raw/vendor.csv`
- `data/raw/purchase_order_header.csv`
- `data/raw/purchase_order_detail.csv`
- `data/raw/product_vendor.csv`
- `data/raw/ship_method.csv`
- `data/raw/product.csv`
- `data/processed/purchasing_consolidated.csv`

El archivo consolidado reúne indicadores a nivel de proveedor. Los CSV originales se conservan por separado para mantener una copia de los datos extraídos.

## 3. Estructura del repositorio

```text
bi-purchasing/
├── data/
│   ├── raw/          # CSV originales extraídos de SQL Server
│   └── processed/    # Dataset consolidado
├── notebooks/
│   └── 01_purchasing_eda.ipynb
├── src/
│   ├── db.py
│   ├── extract_data.py
│   ├── inspect_data.py
│   └── consolidate_data.py
├── .env.example
├── .gitignore
├── requirements.txt
└── README.md
```

La estructura puede actualizarse si el equipo incorpora nuevos scripts o archivos.

## 4. Metodología de trabajo

1. **Conexión y extracción:** conexión a SQL Server y exportación de las tablas seleccionadas a CSV.
2. **Inspección inicial:** revisión del tamaño de las tablas, columnas, tipos de datos y valores nulos.
3. **Consolidación:** relación de los datos y generación de indicadores por proveedor, evitando duplicaciones en los valores de las órdenes.
4. **EDA:** análisis de calidad, estadísticas descriptivas, distribuciones y relaciones relevantes entre variables.
5. **Preparación posterior:** selección y preparación de variables para evaluar la segmentación de proveedores.

La selección definitiva de variables y la interpretación de los grupos se definirán a partir de los resultados del análisis y los acuerdos del equipo.

## 5. Preparación realizada

Hasta el momento se completaron las siguientes actividades:

- Se verificó la conexión desde Python a la base de datos.
- Se extrajeron seis tablas a archivos CSV.
- Se revisó inicialmente el tamaño de las tablas, las columnas y los valores nulos.
- Se generó `purchasing_consolidated.csv` como una primera consolidación a nivel de proveedor.

La consolidación es una base de trabajo inicial y debe validarse antes de utilizarse para análisis estadístico o modelado.

## 6. Resumen del análisis exploratorio de datos (EDA)

El análisis detallado de estructura, calidad y estadística fue desarrollado por Miguel en su notebook **EDA Parte 1 – Estructura, calidad y estadística**. En esta sección se integran los hallazgos principales para dar una visión general del sprint; las tablas, gráficos y cálculos completos permanecen en el notebook de EDA.

### 6.1 Estructura general

El análisis consideró las cuatro tablas principales del esquema Purchasing y el dataset consolidado por proveedor:

| Dataset | Filas | Columnas |
|---|---:|---:|
| `Vendor` | 106 | 5 |
| `PurchaseOrderHeader` | 4.019 | 13 |
| `PurchaseOrderDetail` | 8.853 | 11 |
| `ProductVendor` | 467 | 11 |
| Consolidado por proveedor | 106 | 15 |

El consolidado conserva una fila por proveedor. De los 106 proveedores registrados, 88 tienen información de órdenes que permite calcular las métricas de compra analizadas.

### 6.2 Calidad de los datos y anomalías

- No se encontraron valores faltantes ni filas duplicadas en `Vendor`, `PurchaseOrderHeader` ni `PurchaseOrderDetail`.
- En `ProductVendor` hay 7 valores faltantes en `LastReceiptCost`, 7 en `LastReceiptDate` y 311 en `OnOrderQty` (66,6 %). Estos faltantes deben interpretarse según su significado de negocio; no se utilizan las columnas afectadas para la segmentación propuesta.
- **18 proveedores (16,98 %) no tienen órdenes de compra** y tampoco aparecen asociados a productos en `ProductVendor`. Sus métricas de compra quedan vacías en el consolidado, por lo que no se pueden ubicar en segmentos basados en monto y cantidad de órdenes.
- Se identificó una orden con `TotalDue` igual o menor que cero: la orden 6013, pendiente y sin valor. Se documenta como anomalía para que el equipo decida si debe excluirse de los cálculos.
- Las fechas están almacenadas como texto en los archivos analizados. Deben convertirse a tipo fecha si se realizan análisis temporales.

### 6.3 Distribución y valores atípicos

Los montos de compra presentan asimetría hacia la derecha: la mayoría de los proveedores registra importes menores y unos pocos concentran montos elevados. En `ValorTotalOrdenes`, la media reportada es aproximadamente 801 mil y la mediana aproximadamente 102 mil; el máximo ronda los 5,03 millones.

El análisis mediante IQR identificó valores atípicos en varias variables. Los montos elevados no se consideran automáticamente errores: pueden representar proveedores con una actividad de compra legítimamente alta. Por eso, el notebook de Miguel recomienda conservarlos y revisar su efecto en la segmentación. Algunas variables discretas o casi constantes también aparecen como atípicas por tener un IQR igual a cero, por lo que esos casos no deben eliminarse automáticamente.

### 6.4 Relaciones entre variables y consecuencias para el clustering

La matriz de correlación se calculó sobre los 88 proveedores con métricas de compra disponibles. Los principales hallazgos fueron:

- `ValorTotalOrdenes` y `ValorLineas` tienen correlación de 1,00, por lo que aportan información prácticamente redundante.
- `ProductosComprados` y `ProductosOfrecidos` también tienen correlación de 1,00 en el análisis.
- `NumeroOrdenes` y `ValorTotalOrdenes` presentan correlación baja (r = 0,17), por lo que describen aspectos distintos del comportamiento de compra.
- La cantidad de órdenes varía poco para la mayoría de los proveedores, mientras que los montos muestran una dispersión considerable.

Con base en estos hallazgos, se recomienda evaluar `ValorTotalOrdenes` y `NumeroOrdenes` como variables principales del clustering, de acuerdo con el alcance definido para el sprint. La decisión definitiva corresponde a la etapa de modelado de Molina.

### 6.5 Decisiones y limitaciones

- Mantener identificados los 18 proveedores sin órdenes, pero excluirlos del conjunto que se use para entrenar K-Means, ya que no cuentan con monto ni número de órdenes.
- Conservar los valores altos de compra mientras no exista evidencia de que sean errores.
- Revisar cómo tratar la orden pendiente con `TotalDue` igual a cero antes de cerrar la consolidación.
- Tener en cuenta que incluir órdenes pendientes o rechazadas en los montos podría afectar las métricas. Miguel señala esta limitación para que el equipo la revise antes de interpretar los resultados.

Para el detalle de los cálculos, las tablas de calidad, los gráficos y las estadísticas, consultar el notebook de Miguel.


## 7. Preparación para la segmentación

A partir del EDA de Miguel, la preparación inicial para el clustering queda definida de la siguiente manera. Los resultados finales del modelo se incorporarán cuando Molina complete la evaluación y ejecución de K-Means.

### 7.1 Población de análisis

El consolidado contiene 106 proveedores. De ellos, 18 no tienen órdenes de compra y no cuentan con valores para las métricas de monto y número de órdenes. El notebook de Miguel genera un dataset de trabajo con los **88 proveedores que sí tienen esas métricas**, guardado localmente como `data/processed/purchasing_clean.csv`.

Este archivo se genera al ejecutar el notebook de Miguel. Como la carpeta `data/processed/` está excluida de Git, cada integrante debe generarlo localmente siguiendo el flujo de trabajo acordado; no se debe asumir que estará incluido en el repositorio.

### 7.2 Variables propuestas

Las variables recomendadas para evaluar son:

- `ValorTotalOrdenes`: representa el monto acumulado de las órdenes asociadas a cada proveedor.
- `NumeroOrdenes`: representa la cantidad de órdenes asociadas a cada proveedor.

El EDA reporta una correlación baja entre ambas variables (r = 0,17), lo que sugiere que no son redundantes. Sin embargo, `NumeroOrdenes` presenta poca variación para buena parte de los proveedores, por lo que debe considerarse al interpretar los grupos.

### 7.3 Preparación pendiente antes de entrenar

1. Confirmar con el equipo el tratamiento de órdenes pendientes o rechazadas y de la orden 6013, cuyo importe es cero.
2. Verificar que las dos variables seleccionadas no tengan valores faltantes en el dataset que se utilizará.
3. Aplicar `StandardScaler` antes de K-Means, ya que las variables están en escalas muy diferentes.
4. Evaluar si una transformación logarítmica del monto mejora el tratamiento de su fuerte asimetría; esta decisión debe justificarse y documentarse.
5. Seleccionar el número de clusters mediante el método del codo y el coeficiente de silhouette.
6. Describir los perfiles obtenidos y decidir si los resultados justifican interpretarlos como Micro, Pequeñas, Medianas y Grandes.

**Estado:** preparación conceptual basada en el EDA. La selección definitiva de variables, el número de clusters, las etiquetas y los resultados de K-Means quedan pendientes del notebook de Molina.


## 8. Conclusiones y próximos pasos

La extracción y consolidación permitieron reunir información de 106 proveedores y preparar una base para el análisis de compras. El EDA identificó que 18 proveedores no tienen órdenes registradas, que los montos presentan una distribución muy asimétrica y que algunas variables son redundantes entre sí.

Como siguiente paso, el equipo debe confirmar las decisiones de tratamiento de órdenes pendientes y rechazadas, validar el dataset de 88 proveedores y completar la segmentación con K-Means. Molina documentará la selección del número de clusters, los perfiles y la tabla final de proveedores clasificados.

Una vez disponibles esos resultados, Sebastian integrará las conclusiones del modelo, verificará que las interpretaciones estén respaldadas por los datos y preparará la presentación final del Sprint 1.

**Limitación:** en este notebook aún no se presentan resultados de K-Means ni se afirma que los cuatro segmentos propuestos estén validados; esas conclusiones dependen del modelo que ejecute Molina.
